# conv-windowing-1d — ex2: 1-D conv windowing via Tensor.unfold (the high-level API)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-windowing-1d`. Running the final beacon cell reports progress against the `CNN: 1-D conv windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-1d`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-1d"
DD_SUBTOPIC = "CNN: 1-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 1-D conv windowing — two ways to build the window view

To express a 1-D conv as an einsum, you need a window view `(B, IC, OW, KW)` of the input. There are two natural APIs:

1. **`as_strided`** — explicit shape + stride arithmetic.
2. **`Tensor.unfold(dim, size, step)`** — high-level sliding-window API that returns the same view with sane defaults.

**This drill (ex2) vs ex1.** ex1 built the window view manually via `as_strided` (compute output width, compute strides). ex2 uses the **`Tensor.unfold`** API instead — same view, much less arithmetic, and verifies that the resulting einsum still matches `F.conv1d`. Knowing both lets you choose: `as_strided` for stride !=1 / non-trivial layouts, `unfold` for the common case.

### Exercise 2 — 1-D conv windowing via Tensor.unfold (the high-level API)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `Tensor.unfold(dim, size, step)` to build the `(B, IC, OW, KW)` stride-1 window view of a 1-D input, then verify that contracting it against a kernel via einsum matches `F.conv1d`.
> Keywords: unfold, sliding-window, conv1d, einsum
> ```

**KCs targeted:** `unfold-sliding-window-api`, `windowing-output-width`

Implement `ex2_conv1d_unfold(x, KW)`.

Given a `(B, IC, W)` float tensor `x` and a kernel width `KW`, return the stride-1 window view of shape `(B, IC, OW, KW)` where `OW = W - KW + 1`. Use `Tensor.unfold` rather than `as_strided`.

**Rules.**
1. Call `x.unfold(dimension=-1, size=KW, step=1)` — that's the high-level API.
2. Output shape must be exactly `(B, IC, OW, KW)`.
3. Verify (in the test) that contracting against a kernel via `einsum('b i o k, c i k -> b c o', windows, weight)` reproduces `F.conv1d(x, weight)`.

Inputs:
- `x`: `(B, IC, W)` float tensor with `W >= KW`.
- `KW`: int kernel width.

Output: `(B, IC, OW, KW)` window view (a view, not a copy).

In [ ]:
def ex2_conv1d_unfold(x: Tensor, KW: int) -> Tensor:
    return x.unfold(dimension=-1, size=KW, step=1)


<details><summary>Solution</summary>

```python
def ex2_conv1d_unfold(x: Tensor, KW: int) -> Tensor:
    return x.unfold(dimension=-1, size=KW, step=1)
```

**Why `Tensor.unfold` exists.** `as_strided` is powerful but easy to get wrong — one bad stride and you read off the end of storage (silent corruption). `unfold` is the safer high-level cousin that computes strides for you. Same output, no arithmetic.

**Watch the new axis position.** `x.unfold(dimension=-1, size=KW, step=1)` *appends* the `KW` axis at the end, so a `(B, IC, W)` input becomes `(B, IC, OW, KW)` — exactly the layout you need for the einsum `'b i o k, c i k -> b c o'`. No transpose needed.

**When to still use `as_strided`.** `unfold` only supports a single dimension at a time and forces `step=1` for full-coverage views. For multi-axis windowing (2-D / 3-D convs) or for non-standard strides, you still drop to `as_strided`. ex1 showed the explicit form; ex2 shows the ergonomic shortcut.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()